In [1]:
pip install numba numba-cuda

In [4]:
# vector of 1,000,000 elements and 256 threads per block => 3907 blocks, 8 warps per block
# thread 5 of block 10 => element 10*256 + 5 = 2565

import numpy as np
from numba import cuda

@cuda.jit
def kernel(x, block_idxs, thread_idxs):
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < block_idxs.size:
        block_idxs[i] = cuda.blockIdx.x
        thread_idxs[i] = cuda.threadIdx.x

n = 1_000_000
x = np.arange(n, dtype=np.float32)

# Copy input to the GPU and allocate the output there
d_x = cuda.to_device(x)
d_block_idxs = cuda.device_array_like(x)
d_thread_idxs = cuda.device_array_like(x)

threads_per_block = 256
blocks = (n + threads_per_block - 1) // threads_per_block
kernel[blocks, threads_per_block](d_x, d_block_idxs, d_thread_idxs)   # launch
block_idxs = d_block_idxs.copy_to_host()
thread_idxs = d_thread_idxs.copy_to_host()

print(f"Block: {block_idxs[2565]}")
print(f"Thread: {thread_idxs[2565]}")

Block: 10.0
Thread: 5.0
